# RivaGAN: four inpainting methods

Repository: [https://github.com/DAI-Lab/RivaGAN](https://github.com/DAI-Lab/RivaGAN)

Paper: [https://arxiv.org/abs/1909.01285](https://arxiv.org/abs/1909.01285)

Inspected commit: `efffa72a4ca46d4d5051f6970c96424c2cdab441`

Watermark stage: `post_hoc`
Domain: video

This method can be applied to existing media, so the notebook uses the fixed MOSE sample.

ProPainter, DiffuEraser, EffectErase, and E2FGVI-HQ receive the same authentic watermarked frames and binary mask. One official DiffuEraser execution is used: its internal ProPainter prior is preserved as the ProPainter result, and its final refinement is the DiffuEraser result. Inputs shorter than 22 frames are padded by repeating only the last frame/mask to 24 for that pipeline, then both reported outputs are trimmed to the authentic length. EffectErase applies its released Wan 4n+1 temporal adapter; E2FGVI-HQ uses the authors' official arbitrary-resolution test entry point.

**Status: intentionally deferred for this run, as requested.** Independently, DAI-Lab publishes training code but no pretrained `model.pt`. The notebook remains fully wired for a future run, but it deliberately stops before embedding unless a trained checkpoint and its provenance are supplied. The RivaGAN sample committed by the VideoMark authors is not treated as natively decodable because its matching RivaGAN checkpoint/message are absent.


## Authenticity contract

- Watermark preparation: Load a trained 32-bit RivaGAN model and call its official encode method on the source video.
- Checkpoint status: BLOCKED until a trained model.pt is supplied: the official repository does not publish a pretrained checkpoint.
- Protocol note: RivaGAN can watermark an existing MOSE video, but the attack must not run until the exact trained checkpoint is recorded.
- No result cell fabricates metrics. Native extraction must finish and write its JSON before values are displayed.
- A method without a released spatial detector reports bit recovery only; it is not assigned a proxy localizer.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
from IPython.display import JSON, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/remaining_watermark_methods.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
SCRIPTS = ROOT / "scripts"

def run(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print("+", shlex.join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def require(path, description):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")
    return path

def require_commit(repository, expected):
    actual = subprocess.check_output(["git", "-C", str(repository), "rev-parse", "HEAD"], text=True).strip()
    if actual != expected:
        raise RuntimeError(f"Expected {expected} in {repository}, found {actual}")

def show_results(run_root):
    run_root = Path(run_root)
    for name in ("watermarked.mp4", "aligned-mask.mp4"):
        path = run_root / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    attack = run_root / "attack"
    for name in (
        "input-watermarked.mp4", "gt-mask.mp4", "propainter.mp4", "diffueraser.mp4",
        "effecterase.mp4", "e2fgvi.mp4",
    ):
        path = attack / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    result_json = {
        *run_root.rglob("summary.json"),
        *(attack / name for name in ("sample.json", "removal-pair.json", "effecterase.json", "e2fgvi.json")),
    }
    for path in sorted(path for path in result_json if path.is_file()):
        print(path.relative_to(run_root))
        display(JSON(json.loads(path.read_text())))

print("Repository:", ROOT)


## Canonical one-clip comparison: all four inpainters

The watermark input cannot be prepared, so no remover is selectively presented as if it had run. All four inpainters carry the same explicit status.

| Inpainting method | Status | Reason |
|---|---:|---|
| ProPainter | deferred | deferred by user; no official pretrained model was released |
| DiffuEraser | deferred | deferred by user; no official pretrained model was released |
| EffectErase | deferred | deferred by user; no official pretrained model was released |
| E2FGVI-HQ | deferred | deferred by user; no official pretrained model was released |

Watermark method: **RivaGAN**.


In [ ]:
RIVAGAN = Path("/path/to/RivaGAN")
RIVAGAN_PYTHON = RIVAGAN / ".venv/bin/python"
RIVAGAN_MODEL = Path("/path/to/rivagan/model.pt")  # required; not published upstream
DIFFUERASER = Path("/path/to/DiffuEraser")
DIFFUERASER_PYTHON = DIFFUERASER / ".venv/bin/python"
GPU = "0"
MOSE_VIDEO = ROOT / "evidence/mose-task1/review50/media/02b72b8d/original.mp4"
MOSE_MASK = ROOT / "evidence/mose-task1/review50/media/02b72b8d/gt_mask.mp4"
RUN_ROOT = ROOT / "runs/remaining-watermarks/rivagan/02b72b8d"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
WATERMARKED_VIDEO = RUN_ROOT / "watermarked.mp4"
ALIGNED_MASK = RUN_ROOT / "aligned-mask.mp4"


In [ ]:
require_commit(RIVAGAN, "efffa72a4ca46d4d5051f6970c96424c2cdab441")
require(RIVAGAN_PYTHON, "RivaGAN Python")
require(RIVAGAN_MODEL, "trained RivaGAN model.pt")
require(MOSE_VIDEO, "MOSE video")
require(MOSE_MASK, "MOSE mask")


In [ ]:
run([
    RIVAGAN_PYTHON, SCRIPTS / "rivagan_video.py", "embed",
    "--source", RIVAGAN, "--checkpoint", RIVAGAN_MODEL,
    "--input-video", MOSE_VIDEO, "--input-mask", MOSE_MASK,
    "--output-video", WATERMARKED_VIDEO, "--output-mask", ALIGNED_MASK,
])


## Run all four inpainting methods on the frozen input

The preparation cell freezes hashes, frame count, dimensions, FPS, mask foreground counts, provenance, and any short-video padding. Every remover below consumes that frozen input and mask.


In [ ]:
ATTACK_RUN = RUN_ROOT / "attack"
if not (ATTACK_RUN / "sample.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "prepare_watermark_removal_sample.py", "prepare",
        "--source-video", WATERMARKED_VIDEO, "--source-mask", ALIGNED_MASK,
        "--output", ATTACK_RUN, "--watermark-method", "RivaGAN",
        "--sample-id", "02b72b8d", "--sample-source", "MOSE fixed sample",
        "--source-kind", "mose_post_hoc",
    ])
else:
    print("Prepared sample already exists:", ATTACK_RUN / "sample.json")


In [ ]:
if not (ATTACK_RUN / "removal-pair.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "run_watermark_removal_pair.py",
        "--run", ATTACK_RUN, "--diffueraser", DIFFUERASER,
        "--diffueraser-python", DIFFUERASER_PYTHON,
        "--device", GPU, "--seed", "2025",
    ])
else:
    print("Both removal outputs already exist:", ATTACK_RUN / "removal-pair.json")


In [ ]:
EFFECTERASE = Path(os.environ.get("EFFECTERASE", "/mnt/data_fast/smayan/research/EffectErase"))
EFFECTERASE_PYTHON = Path(os.environ.get("EFFECTERASE_PYTHON", str(EFFECTERASE / ".venv/bin/python")))
E2FGVI = Path(os.environ.get("E2FGVI", "/mnt/data_fast/smayan/research/E2FGVI"))
E2FGVI_CHECKPOINT = Path(os.environ.get(
    "E2FGVI_CHECKPOINT", str(E2FGVI / "release_model/E2FGVI-HQ-CVPR22.pth")
))
E2FGVI_PYTHON = Path(os.environ.get("E2FGVI_PYTHON", str(DIFFUERASER_PYTHON)))

if not (ATTACK_RUN / "effecterase.json").is_file():
    effect_env = os.environ.copy(); effect_env["CUDA_VISIBLE_DEVICES"] = GPU
    run([
        EFFECTERASE_PYTHON, SCRIPTS / "run_prepared_effecterase.py",
        "--source", EFFECTERASE, "--run", ATTACK_RUN, "--device", GPU,
    ], env=effect_env)
else:
    print("EffectErase output already exists:", ATTACK_RUN / "effecterase.json")

if not (ATTACK_RUN / "e2fgvi.json").is_file():
    run([
        E2FGVI_PYTHON, SCRIPTS / "run_prepared_e2fgvi.py",
        "--source", E2FGVI, "--checkpoint", E2FGVI_CHECKPOINT,
        "--run", ATTACK_RUN, "--python", E2FGVI_PYTHON, "--device", GPU,
    ])
else:
    print("E2FGVI-HQ output already exists:", ATTACK_RUN / "e2fgvi.json")


In [ ]:
run([
    RIVAGAN_PYTHON, SCRIPTS / "rivagan_video.py", "detect",
    "--source", RIVAGAN, "--checkpoint", RIVAGAN_MODEL,
    "--output", RUN_ROOT / "detection",
    "--case", f"original={ATTACK_RUN / 'input-watermarked.mp4'}",
    "--case", f"propainter={ATTACK_RUN / 'propainter.mp4'}",
    "--case", f"diffueraser={ATTACK_RUN / 'diffueraser.mp4'}",
    "--case", f"effecterase={ATTACK_RUN / 'effecterase.mp4'}",
    "--case", f"e2fgvi={ATTACK_RUN / 'e2fgvi.mp4'}",
])


In [ ]:
show_results(RUN_ROOT)
